In [1]:
import requests
import json
import csv

print('cityair tracker')

cities = [
    ('ahmedabad', 23.03, 72.58),
    ('surat', 21.17, 72.83),
    ('vadodara', 22.31, 73.18),
    ('rajkot', 22.30, 70.80),
    ('mumbai', 19.08, 72.88),
    ('delhi', 28.61, 77.21),
    ('pune', 18.52, 73.86),
    ('bengaluru', 12.97, 77.59),
    ('chennai', 13.08, 80.27),
    ('kolkata', 22.57, 88.36)
]

url = 'https://air-quality-api.open-meteo.com/v1/air-quality'
hourly_raw = {}

for city, lat, lon in cities:
    resp = requests.get(
        url,
        params={
            'latitude': lat,
            'longitude': lon,
            'hourly': 'pm2_5',
            'forecast_days': 1,
            'timezone': 'Asia/Kolkata'
        },
        timeout=10
    ).json()
    vals = resp['hourly']['pm2_5'][:24]
    hourly_raw[city] = vals

for c, v in hourly_raw.items():
    print(c, len(v))

cityair tracker
ahmedabad 24
surat 24
vadodara 24
rajkot 24
mumbai 24
delhi 24
pune 24
bengaluru 24
chennai 24
kolkata 24


In [2]:
rows = []

for city, vals in hourly_raw.items():
    first_pm25 = vals[0]
    last_pm25 = vals[23]

    if first_pm25 is None or first_pm25 == 0 or last_pm25 is None:
        change_pct = None
    else:
        change_pct = round((last_pm25 - first_pm25) / first_pm25 * 100, 2)

    valid_vals = [x for x in vals if x is not None]
    high_pm25 = max(valid_vals) if valid_vals else None
    low_pm25 = min(valid_vals) if valid_vals else None
    avg_pm25 = round(sum(valid_vals) / len(valid_vals), 2) if valid_vals else None

    rows.append({
        'city': city,
        'first_pm25': first_pm25,
        'last_pm25': last_pm25,
        'change_pct': change_pct,
        'high_pm25': high_pm25,
        'low_pm25': low_pm25,
        'avg_pm25': avg_pm25
    })

fieldnames = ['city', 'first_pm25', 'last_pm25', 'change_pct', 'high_pm25', 'low_pm25', 'avg_pm25']
with open('aqi_tracker.csv', 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(rows)

print('saved rows:', len(rows))

saved rows: 10


In [3]:
with open('aqi_hourly_raw.json', 'w', encoding='utf-8') as f:
    json.dump(hourly_raw, f, indent=2)

with open('aqi_hourly_raw.json', 'r', encoding='utf-8') as f:
    loaded_json = json.load(f)

with open('aqi_tracker.csv', 'r', newline='', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    loaded_csv = list(reader)
    csv_cols = reader.fieldnames

print('csv rows:', len(loaded_csv), 'csv cols:', len(csv_cols))
print('json city count:', len(loaded_json), 'all 24:', all(len(v) == 24 for v in loaded_json.values()))

sorted_rows = sorted(loaded_csv, key=lambda r: float(r['avg_pm25']), reverse=True)
for r in sorted_rows:
    print(f"{r['city']:<12} avg={r['avg_pm25']} high={r['high_pm25']} low={r['low_pm25']} chg={r['change_pct']}")

csv rows: 10 csv cols: 7
json city count: 10 all 24: True
delhi        avg=170.18 high=212.4 low=102.8 chg=-1.35
kolkata      avg=101.13 high=126.0 low=60.8 chg=-2.23
ahmedabad    avg=64.83 high=81.0 low=41.0 chg=2.34
mumbai       avg=54.87 high=68.0 low=35.2 chg=-1.97
vadodara     avg=53.6 high=65.5 low=35.8 chg=-1.72
surat        avg=48.48 high=61.4 low=31.5 chg=-1.15
rajkot       avg=45.28 high=56.0 low=29.8 chg=-1.82
pune         avg=40.02 high=50.2 low=25.5 chg=-1.14
chennai      avg=37.68 high=46.8 low=24.6 chg=-1.94
bengaluru    avg=33.42 high=42.2 low=21.2 chg=-1.37
